In [ ]:
from pathlib import Path
PROJECT_ROOT = next(
    p for p in Path.cwd().parents if p.name == "LaboroTomato"
)


import json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt

In [2]:
def build_model(name: str, num_classes: int) -> nn.Module:
    name = name.lower()
    if name == "mobilenet_v2":
        weights = models.MobileNet_V2_Weights.IMAGENET1K_V1
        m = models.mobilenet_v2(weights=weights)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)
        return m
    if name == "efficientnet_v2_s":
        weights = models.EfficientNet_V2_S_Weights.IMAGENET1K_V1
        m = models.efficientnet_v2_s(weights=weights)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)
        return m
    raise ValueError(f"Unknown model: {name}")

In [3]:
def main():
    RUN_DIR = PROJECT_ROOT / "runs_cpu_local" / "baseline_mobilenetv2" / "20251228-003415"
    DATA_ROOT = PROJECT_ROOT / "dataset" / "cropped"

    BEST_CKPT = RUN_DIR / "model_best.pt"
    CLASS_MAP = RUN_DIR / "class_to_idx.json"

    assert BEST_CKPT.exists(), f"Missing checkpoint: {BEST_CKPT}"
    assert CLASS_MAP.exists(), f"Missing class map: {CLASS_MAP}"
    assert (
        DATA_ROOT / "test").exists(), f"Missing test folder: {DATA_ROOT / 'test'}"

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Device:", device)

    ckpt = torch.load(BEST_CKPT, map_location="cpu")
    cfg = ckpt.get("config", {})
    model_name = cfg.get("model_name", "mobilenet_v2")
    image_size = int(cfg.get("image_size", 224))
    batch_size = int(cfg.get("batch_size", 64))

    with open(CLASS_MAP, "r", encoding="utf-8") as f:
        class_to_idx = json.load(f)

    idx_to_class = {v: k for k, v in class_to_idx.items()}
    class_names = [idx_to_class[i] for i in range(len(idx_to_class))]
    num_classes = len(class_names)

    print("Model:", model_name)
    print("Classes:", class_names)

    # -----------------------------
    # Dataset / loader (test only)
    # -----------------------------
    imagenet_mean = (0.485, 0.456, 0.406)
    imagenet_std = (0.229, 0.224, 0.225)

    eval_tfms = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(imagenet_mean, imagenet_std),
    ])

    test_ds = datasets.ImageFolder(
        root=str(DATA_ROOT / "test"), transform=eval_tfms)
    test_loader = DataLoader(
        test_ds, batch_size=batch_size, shuffle=False,
        num_workers=0,  # Windows-safe
        pin_memory=True
    )

    print("ImageFolder classes (alphabetical):", test_ds.classes)
    print("Saved class_to_idx:", class_to_idx)

    # -----------------------------
    # Model
    # -----------------------------
    model = build_model(model_name, num_classes=num_classes)
    model.load_state_dict(ckpt["model_state"])
    model.to(device)
    model.eval()

    # -----------------------------
    # Inference
    # -----------------------------
    all_y = []
    all_pred = []

    with torch.no_grad():
        for x, y in test_loader:
            x = x.to(device, non_blocking=True)
            logits = model(x)
            pred = logits.argmax(dim=1).cpu().numpy()

            all_pred.append(pred)
            all_y.append(y.numpy())

    y_true = np.concatenate(all_y)
    y_pred = np.concatenate(all_pred)

    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    cm = confusion_matrix(y_true, y_pred)

    print("\n=== Test split ===")
    print(f"Accuracy : {acc:.4f}")
    print(f"Macro F1 : {macro_f1:.4f}\n")

    print("Classification report:")
    print(classification_report(y_true, y_pred,
          target_names=class_names, digits=4))

    print("Confusion matrix (rows=true, cols=pred):\n", cm)

    # -----------------------------
    # Save artifacts
    # -----------------------------
    RUN_DIR.mkdir(parents=True, exist_ok=True)

    # Confusion matrix plot
    fig = plt.figure(figsize=(6, 5))
    plt.imshow(cm)
    plt.title("Confusion Matrix (Test)")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.xticks(range(num_classes), class_names, rotation=30, ha="right")
    plt.yticks(range(num_classes), class_names)

    for i in range(num_classes):
        for j in range(num_classes):
            plt.text(j, i, str(cm[i, j]), ha="center", va="center")

    plt.tight_layout()
    cm_path = RUN_DIR / "confusion_matrix.png"
    fig.savefig(cm_path, dpi=200)
    plt.close(fig)

    metrics = {
        "accuracy": float(acc),
        "macro_f1": float(macro_f1),
        "class_names": class_names,
        "confusion_matrix": cm.tolist(),
        "checkpoint": str(BEST_CKPT),
        "data_root": str(DATA_ROOT),
        "split": "test",
    }
    metrics_path = RUN_DIR / "metrics.json"
    with open(metrics_path, "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2)

    print(f"\nSaved: {cm_path}")
    print(f"Saved: {metrics_path}")

In [4]:
if __name__ == "__main__":
    main()

Device: cpu
Model: mobilenet_v2
Classes: ['fully_ripened', 'green', 'half_ripened']
ImageFolder classes (alphabetical): ['fully_ripened', 'green', 'half_ripened']
Saved class_to_idx: {'fully_ripened': 0, 'green': 1, 'half_ripened': 2}

=== Test split ===
Accuracy : 0.8197
Macro F1 : 0.7435

Classification report:
               precision    recall  f1-score   support

fully_ripened     0.7716    0.6130    0.6832       540
        green     0.8319    0.9584    0.8907      1777
 half_ripened     0.8069    0.5534    0.6565       506

     accuracy                         0.8197      2823
    macro avg     0.8035    0.7082    0.7435      2823
 weighted avg     0.8159    0.8197    0.8090      2823

Confusion matrix (rows=true, cols=pred):
 [[ 331  174   35]
 [  42 1703   32]
 [  56  170  280]]

Saved: c:\Users\User\Desktop\LaboroTomato\runs_cpu_local\baseline_mobilenetv2\20251228-003415\confusion_matrix.png
Saved: c:\Users\User\Desktop\LaboroTomato\runs_cpu_local\baseline_mobilenetv2\202512

*Summary (natural background baseline):*\
***While the baseline MobileNetV2 achieved an accuracy of 81.97% on the test split, the macro-averaged F1-score dropped to 74.35%, indicating uneven performance across ripeness stages. In particular, the model exhibited strong recall for green tomatoes (95.84%), while recall for half-ripened (55.34%) and fully ripened (61.30%) samples was substantially lower, with these classes frequently misclassified as green. This asymmetric error pattern motivates a deeper reliability and calibration analysis.***

In [1]:
!jupyter nbconvert --to html 03_eval_performance.ipynb


[NbConvertApp] Converting notebook 03_eval_performance.ipynb to html
[NbConvertApp] Writing 301931 bytes to 03_eval_performance.html
